In [ ]:
library(Seurat)
library(sctransform)
library(tidyverse)
library(repr)
library(patchwork)
library(biomaRt)
library(parallel)
library(DoubletFinder)
library(ggpubr)
library(ggsci)
library(gprofiler2)

library(future)
plan("multisession", workers = 10)
options(future.globals.maxSize= 60e+09)
options(Seurat.object.assay.version = "v3")

# 1. run -- C4007

In [ ]:
## 
samples = c("LLT-1","LLT-2","RLT","ThyT","LvT-1","LvT-2","OvaT","IntT")
files=c("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-LLT-1/outs/filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-LLT-2/outs/filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Chest/outs/per_sample_outs/RLT/count/sample_filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Chest/outs/per_sample_outs/ThyT/count/sample_filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Abdo/outs/per_sample_outs/LvT-1/count/sample_filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Abdo/outs/per_sample_outs/LvT-2/count/sample_filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Abdo/outs/per_sample_outs/OvaT/count/sample_filtered_feature_bc_matrix",
        "/syn2/zhaolian/3.JiLab/results/2.scRNAseq/1.cellranger/C4007-Abdo/outs/per_sample_outs/IntT/count/sample_filtered_feature_bc_matrix")
names(files)=samples
files


In [ ]:
# load rawcount
rawCounts=Read10X(data.dir = files)
# create seurat object
scdata=CreateSeuratObject(rawCounts$`Gene Expression`,project = "C4007")
scdata
colnames(scdata)[1:5]
rownames(scdata)[1:5]
scdata@meta.data=rename(scdata@meta.data,sampleID=orig.ident)
head(scdata@meta.data)

In [ ]:
table(scdata@meta.data$sampleID)

### mt,ribo

In [ ]:
scdata[["percent.mt"]] = PercentageFeatureSet(scdata, pattern = "^mt-")
scdata[["percent.ribo"]] = PercentageFeatureSet(scdata, pattern = "^Rp[sl]")
options(repr.plot.width = 15, repr.plot.height = 4)
metadataPlot_raw=VlnPlot(scdata, group.by = "sampleID", features = c("nFeature_RNA", "nCount_RNA", "percent.mt","percent.ribo"), pt.size = 0, ncol = 4)
metadataPlot_raw

### QC

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
nFeatureplot=hist(scdata@meta.data$nFeature_RNA,breaks = 1000)#,xlim = c(0,4000),ylim = c(0,400)
pct.mt=hist(scdata@meta.data$percent.mt,breaks = 1000)
nCountplot=hist(scdata@meta.data$nCount_RNA,breaks = 1000)#,xlim = c(0,6000),ylim = c(0,1500)
nCountplot=hist(scdata@meta.data$nCount_RNA,breaks = 1000,xlim = c(0,60000))#,xlim = c(0,6000),ylim = c(0,1500)
pct.ribo=hist(scdata@meta.data$percent.ribo,breaks = 1000)

In [ ]:
# filter
scFiltered = subset(scdata, subset = nFeature_RNA > 500 & nFeature_RNA < 8000 & percent.mt < 5 & nCount_RNA > 4000 & nCount_RNA < 60000)
scdata
scFiltered

# sc <- subset(sc, subset = nFeature_RNA > 500 & nFeature_RNA < 7500 & percent.mt < 20)
options(repr.plot.width = 5, repr.plot.height = 5)
FeatureScatter(scdata,feature1 = 'nCount_RNA',feature2 = 'nFeature_RNA')
FeatureScatter(scFiltered,feature1 = 'nCount_RNA',feature2 = 'nFeature_RNA')

In [ ]:
FeatureScatter(scFiltered,feature1 = 'nCount_RNA',feature2 = 'percent.mt')

### cell cycle

In [ ]:
options(future.globals.maxSize = 60e+09)

In [ ]:
library(gprofiler2)
mus.s = gorth(cc.genes.updated.2019$s.genes, source_organism = "hsapiens", target_organism = "mmusculus")$ortholog_name
mus.g2m = gorth(cc.genes.updated.2019$g2m.genes, source_organism = "hsapiens", target_organism = "mmusculus")$ortholog_name
mus.s
mus.g2m
#merge
mus.cellcycle.genes=c(mus.g2m,mus.s)

#rownames(scFiltered) include all 91 genes 
length(intersect(mus.cellcycle.genes,rownames(scFiltered)))

# calculate cell cycle score
# scFiltered <- JoinLayers(scFiltered)
scFiltered=scFiltered %>% NormalizeData(verbose = FALSE) %>% ScaleData(verbose = FALSE,features = mus.cellcycle.genes)

scFiltered= CellCycleScoring(scFiltered, s.features = mus.s, g2m.features = mus.g2m,set.ident = TRUE)
options(repr.plot.width = 16, repr.plot.height = 4)
RidgePlot(scFiltered, 
          features = c("Pcna", "Mcm6", "Top2a", "Mki67"), 
          cols = pal_npg("nrc", alpha = 0.7)(3),
          ncol = 4)

scFiltered = RunPCA(scFiltered, features = c(mus.s, mus.g2m),verbose = F, nfeatures.print = 10)

options(repr.plot.width = 5, repr.plot.height = 5)
DimPlot(scFiltered, cols = pal_npg("nrc", alpha = 0.7)(3))

In [ ]:
# # scFiltered_cellcycle <- JoinLayers(scFiltered)
# scFiltered_cellcycle <- ScaleData(scFiltered, vars.to.regress = c("S.Score", "G2M.Score"), features = rownames(scFiltered))
# scFiltered_cellcycle <- RunPCA(scFiltered_cellcycle, features = c(s.genes, g2m.genes), nfeatures.print = 10)
# # DimPlot(scFiltered_cellcycle, reduction = "pca", group.by = "Phase")
# DimPlot(scFiltered_cellcycle, cols = pal_npg("nrc", alpha = 0.7)(3))

In [ ]:
scFiltered

In [ ]:
scFiltered=DietSeurat(scFiltered,assays = "RNA")
saveRDS(scFiltered,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scFiltered.rds")
write.table(scFiltered@meta.data[,1:2], file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scFiltered_metadata.txt",sep="\t", quote=FALSE)



# 2. remove doublet

- heterotypic doublet: two different cells, different expression profile.- 
homotypic doublet: two cells of same/similar cell type, similar expression profile.
- 
pN: # of artificial doublets, default value 0.25 but authors have found DoubletFinder performance is not dependent on th value.
- e
pK: the neighborhood size (pK) used to compute the number of artificial nearest neighbors, DoubletFinder performance dependent on this va.
- ue
exp: # of expected real doub.l- ets
DoubletFinder should not be run on aggregated data and should be run on a per sample .b- asis
DoubletFinder should be run after filtering out low quality. - cells
10x doubl.et rate
Multiplet rate (%)|# of Cells Loaded|# of Cells R.ecovered
0.40%—————|800——————|00———————
0.80%—————|1,600—————|1000——————–
1.60%—————|3,200—————|2,000——————-
2.30%—————|4,800—————|3,000——————-
3.10%—————|6,400—————|4,000——————-
3.90%—————|8,000—————|5,000——————-
4.60%—————|9,600—————|6,000——————-
5.40%—————|11,200———-|7,000——————-
6.10%—————|12,800———-|8,000——————-0
6.90%—————|14,40————-|9,000——————-
7.60%—————|16,000————-|10,000——————————-|10,000——————

In [ ]:
scFiltered=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scFiltered.rds")
scFiltered@meta.data$sampleID <- paste0("C4007_",scFiltered@meta.data$sampleID)
scFiltered[["mouse"]]="C4007"
batch_info <- read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/3.integrated/cellNum_batch.txt",header = T)
sampleType=as.character(scFiltered$sampleID)
for(i in unique(sampleType)){names(sampleType)[which(sampleType==i)] <- batch_info[batch_info$sampleID==i,]$batch}
scFiltered[['batch']] <- names(sampleType)
head(scFiltered@meta.data)
table(scFiltered$sampleID)
table(scFiltered$batch)

In [ ]:
sampleList=SplitObject(scFiltered,split.by = "batch")
sampleList

In [ ]:
# sampleList=SplitObject(scFiltered,split.by = "sampleID")

for (i in seq_along(sampleList)) {
    sample=sampleList[[i]]
    ## Pre-process Seurat object (sctransform)
    sample=SCTransform(sample, vst.flavor = "v2", verbose = FALSE) %>%
        RunPCA(verbose = FALSE) %>%
        RunUMAP(dims = 1:50,verbose = FALSE) %>%
        FindNeighbors(reduction = "pca",dims = 1:50,verbose = FALSE) %>%
        FindClusters(resolution = 0.3, verbose = FALSE)
    ## pK Identification 
    sweep.res.list = paramSweep(sample, PCs = 1:30, sct = T) #_v3
    sweep.stats = summarizeSweep(sweep.res.list, GT = FALSE)
    bcmvn = find.pK(sweep.stats)
    opt_pK = as.numeric(as.vector(bcmvn$pK[which.max(bcmvn$BCmetric)]))
    
    
    ## Homotypic Doublet Proportion Estimate
    doubletRate=0.05 #ncol(sample)*8*1e-6
    homotypic.prop = modelHomotypic(sample$seurat_clusters)           
    nExp_poi = round(doubletRate*nrow(sample@meta.data)) 
    nExp_poi.adj = round(nExp_poi*(1-homotypic.prop))
    ## Run DoubletFinder
    sample <- doubletFinder(seu = sample, #_v3 
                               PCs = 1:30,
                               pN=0.25,
                               pK = opt_pK,
                               nExp = nExp_poi.adj,
                               sct=T)
    ## change column name
    colnames(sample@meta.data)[ncol(sample@meta.data)-1] = "pANN"
    colnames(sample@meta.data)[ncol(sample@meta.data)] = "DF.classifications"
    ## get results
    sampleList[[i]]=sample
}

In [ ]:
scdataDF=merge(sampleList[[1]],y=c(sampleList[[2]],sampleList[[3]],sampleList[[4]]))
table(scdataDF@meta.data$sampleID,scdataDF@meta.data$DF.classifications)

DefaultAssay(scdataDF)="RNA"
scdataDF=DietSeurat(scdataDF,assays = "RNA")
saveRDS(scdataDF,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scdataDF.rds")
write.table(scdataDF@meta.data[,1:2], file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scdataDF_metadata.txt",sep="\t", quote=FALSE)
write.table(scdataDF@meta.data[,1:2], file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scdataDF_metadata.txt",sep="\t", quote=FALSE)


## 3. remove doublet and immune clusters

In [ ]:
scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/2.scFiltered/C4007_scdataDF.rds")
table(scdata$sampleID,scdata$DF.classifications)
nointeg=scdata%>%
    NormalizeData(verbose = F)%>%
    FindVariableFeatures(verbose = F)%>%
    ScaleData(verbose = F)%>%
    RunPCA(verbose = F)%>%
    FindNeighbors(reduction = "pca", dims = 1:30,verbose = F)%>%
    FindClusters(resolution = 0.2,verbose = F)
nointeg = RunUMAP(nointeg, reduction = "pca", dims = 1:30, verbose = FALSE)

options(repr.plot.width = 5, repr.plot.height = 4)
nointeg_umap=DimPlot(nointeg, reduction = "umap", label = T)+theme(axis.title = element_text(size = 8)) #+NoLegend()
nointeg_classification=DimPlot(nointeg, reduction = "umap", group.by = "DF.classifications", label = F)+theme(axis.title = element_text(size = 8))
nointeg_sampleType=DimPlot(nointeg, reduction = "umap", group.by = "sampleID", label = T)+theme(axis.title = element_text(size = 8))

# genes_interest <- c('Ascl1', 'Neurod1', 'Yap1', 'Pou2f3','Epcam','Ncam1','Trp53','Rb1','Ptprc')
# options(repr.plot.width = 10, repr.plot.height = 9)
# FeaturePlot(nointeg, features = genes_interest)
options(repr.plot.width = 15, repr.plot.height = 4)
ggarrange(nointeg_umap,nointeg_classification,nointeg_sampleType, ncol=3,nrow=1)

options(repr.plot.width = 8, repr.plot.height = 4)
VlnPlot(nointeg,features=c("Ptprc"),ncol=1)